# OpenLearn-AI OCR Benchmark — External GPU Smoke (Docling × Misraj × 20)

Infrastructure validation only: run the **existing** benchmark, unchanged, on a Colab GPU.

- One architecture, one Engine abstraction. Colab is an execution environment, not a benchmark layer.
- Expected wall time: ~5 min setup + model download (~1 GB) + ~1 min inference on T4.

In [ ]:
#@title Configuration { display-mode: "form" }
REPO_URL = "https://github.com/MuhammadSeyam/OpenLearn-AI.git"  #@param {type:"string"}
PINNED_COMMIT = "ef7ce427a43d99928acd0f4991f4dfb1e7551772"  #@param {type:"string"}
# NOTE: the pinned commit must contain the committed benchmark implementation.
# See docs/external-gpu-workflow.md step 0.
DRIVE_DIR = "/content/drive/MyDrive/ocrbench"  #@param {type:"string"}
ARCHIVE_NAME = "ocrbench-misraj-data-v1.tar.gz"  #@param {type:"string"}
ARCHIVE_SHA256 = "b66f8e9af44197bf65c2ee0f1c684744e16495390cf87c9fada7fc76af03f7b0"  #@param {type:"string"}
LIMIT = 20  #@param {type:"integer"}

## 1–3. Runtime information & CUDA gate

In [ ]:
!nvidia-smi
import sys, shutil, platform
print("python:", platform.python_version())
print("colab python binary:", sys.executable)
assert shutil.which("nvidia-smi"), "No GPU runtime attached — Runtime > Change runtime type > GPU"
print("GPU runtime detected")

In [ ]:
import torch  # Colab system torch is only used for this environment gate
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f"Total VRAM: {props.total_memory / 1e9:.1f} GB | compute capability: {props.major}.{props.minor}")
print("CUDA (torch build):", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA unavailable — GPU-first policy forbids CPU fallback"
x = torch.randn(512, 512, device="cuda")
assert float((x @ x).sum()) != 0.0
print("CUDA tensor op OK")

## 4–7. Clone repository at pinned commit

In [ ]:
%cd /content
![ -d OpenLearn-AI ] && echo "repo already cloned"
![ -d OpenLearn-AI ] || git clone {REPO_URL}
%cd /content/OpenLearn-AI
!git checkout {PINNED_COMMIT}
!git rev-parse HEAD
BENCH = "/content/OpenLearn-AI/experiments/OCR/ocr-benchmark"
%cd {BENCH}
!ls src/ocrbench

## 8–10. uv-managed Python 3.12 environment + Docling

In [ ]:
%pip install -q uv
!uv --version
# Bootstrap an isolated Python 3.12 regardless of the host image version,
# then create the project env strictly from the repository's own configuration.
!uv python install 3.12
!uv venv --python 3.12 .venv
!uv sync --frozen
!uv pip install "docling>=2.0"          # ad-hoc engine dependency (project policy: engines stay out of pyproject)
!.venv/bin/python -c "import ocrbench, docling, torch; print('ocrbench OK | docling', docling.__version__, '| torch cuda:', torch.cuda.is_available())"

## 11–14. Canonical dataset archive (uploaded ONCE to Drive)

Build locally once: `bash scripts/build_misraj_archive.sh`, then upload
`dist/ocrbench-misraj-data-v1.tar.gz` to the Drive folder.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
!ls -lh "{DRIVE_DIR}/{ARCHIVE_NAME}"

In [ ]:
import hashlib, tarfile
from pathlib import Path

archive = Path(DRIVE_DIR) / ARCHIVE_NAME
h = hashlib.sha256(archive.read_bytes()).hexdigest()
print("archive sha256:", h)
assert h == ARCHIVE_SHA256, "ARCHIVE HASH MISMATCH — do not extract; rebuild/re-upload the archive"

!tar -xzf "{archive}" -C "{BENCH}"

# integrity record check (hash + byte size per file)
for line in (Path(BENCH) / "configs/datasets/misraj_DATA_MANIFEST.sha256").read_text().splitlines():
    expected_hash, expected_size, rel = line.split()
    f = Path(BENCH) / rel
    assert f.is_file(), f"missing after extraction: {rel}"
    assert f.stat().st_size == int(expected_size), f"size mismatch: {rel}"
    digest = hashlib.sha256(f.read_bytes()).hexdigest()
    assert digest == expected_hash, f"hash mismatch: {rel}"
    print("verified:", rel, f"({expected_size} bytes)")
print("dataset verification PASSED")

## 15. Run the EXISTING benchmark command

In [ ]:
!.venv/bin/python -m ocrbench.run.run_text --limit {LIMIT} --engine docling

## 16–17. Locate and validate the timestamped result directory

In [ ]:
import json
from pathlib import Path

runs = sorted((Path(BENCH) / "results/formal/misraj/docling").iterdir())
run_dir = runs[-1]
print("result directory:", run_dir)

raw_files = list((run_dir / "raw_outputs").glob("*.json"))
metrics = json.loads((run_dir / "metrics.json").read_text())
for artifact in ["raw_outputs", "metrics.json", "config.yaml", "run_log.md"]:
    assert (run_dir / artifact).exists(), f"missing artifact: {artifact}"
assert metrics["sample_count"] == LIMIT
assert metrics["successful_samples"] + metrics["failed_samples"] == LIMIT
assert len(raw_files) == LIMIT
assert metrics["accelerator_device"] == "cuda", "GPU-first violation"
assert metrics["micro"]["cer_normalized"] >= 0.0
print(f"validated: {LIMIT} raw outputs | ok={metrics['successful_samples']} "
      f"| micro CER norm={metrics['micro']['cer_normalized']:.4f} | device=cuda")

## 18–20. Archive the result and persist to Drive

In [ ]:
artifact = f"{DRIVE_DIR}/external_result_{run_dir.name}.tar.gz"
!tar -czf "{artifact}" -C "{run_dir.parent}" "{run_dir.name}"
!sha256sum "{artifact}"
print("ARTIFACT WRITTEN TO:", artifact)

## Next: local validation
Download the artifact from Drive, then locally:
```
mkdir -p results/formal/misraj/docling/<same-timestamp>
tar -xzf external_result_<timestamp>.tar.gz -C results/formal/misraj/docling/
```
Compare structure against `results/formal/misraj/docling/20260825-184822/`
(same four artifact classes; selected_sample_ids must be identical — same loader order).
See `docs/external-gpu-workflow.md`.